# Data Exploration: Shopping Cart Trajectories

In this notebook, we explore the shopping cart data located in `../data/raw`. The data consists of `x`, `y` coordinates, timestamps, and a quality metric `q`.

Goals:
- Explore raw data quality and drift.
- Filter out artifacts (charging stations, outside shop bounds) to assess data usability.
- Re-analyze drift on the cleaned data.
- Assess positioning accuracy on stationary devices (standard deviation).

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_style('whitegrid')

In [ ]:
# Load a sample of the data 
raw_data_dir = '../data/raw'
all_files = glob.glob(os.path.join(raw_data_dir, 'node_*.csv'))

import gc
dfs = []
for f in all_files[:3]: # Load 3 files minimum to see drift vs q but save memory
    df = pd.read_csv(f, usecols=['node_id', 'timestamp', 'x', 'y', 'q'], dtype={'node_id': 'int32', 'x': 'float32', 'y': 'float32', 'q': 'uint8'})
    dfs.append(df)
    
data = pd.concat(dfs, ignore_index=True)
del dfs
gc.collect()
data['timestamp'] = pd.to_datetime(data['timestamp'])
data = data.sort_values(by=['node_id', 'timestamp'])
print("Raw Data Shape:", data.shape)

## 1. Raw Data Exploration (Before Filtering)
Let's check for missing values, the distribution of `q` (quality), and the drift on raw data.

In [ ]:
print("Missing values:\n", data.isnull().sum())
print("\nData Describe:\n", data.describe())

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data['q'], bins=50, kde=True)
plt.title('Distribution of Quality (q) - Raw Data')
plt.xlabel('q')
plt.show()

In [ ]:
# Calculate time difference and distance between consecutive points for each node
data['time_diff'] = data.groupby('node_id')['timestamp'].diff().dt.total_seconds()
data['x_diff'] = data.groupby('node_id')['x'].diff()
data['y_diff'] = data.groupby('node_id')['y'].diff()
data['distance'] = np.sqrt(data['x_diff']**2 + data['y_diff']**2)

# Calculate speed (units per second) - approximating drift
data['speed'] = data['distance'] / data['time_diff']

plt.figure(figsize=(10, 5))
sns.histplot(data[data['distance'] < data['distance'].quantile(0.95)]['distance'], bins=50)
plt.title('Distribution of Distance between Consecutive Points (Raw Data, Zoomed to 95th Percentile)')
plt.xlabel('Distance')
plt.show()

In [ ]:
# Raw Drift vs Q
plt.figure(figsize=(10, 6))
sns.scatterplot(x='q', y='distance', data=data, alpha=0.3)
plt.title('Raw Data: Scatter Plot of Quality (q) vs Distance (Drift)')
plt.xlabel('Quality (q)')
plt.ylabel('Distance')
plt.yscale('log') 
plt.show()

## 2. Positioning Usability (Hyvyys/Käytettävyys) - Applying Filters
We count how many points are actual shop visits versus artifacts (out of bounds or charging stations without signal at 0,0).

In [ ]:
data['is_charging_station'] = (data['x'] == 0) & (data['y'] == 0)

# Määritellään kaupan fyysiset koordinaattirajat
shop_bounds = (
    ((data['x'] > 350) & (data['y'] < 3000) & (data['x'] < 1500)) |
    ((data['x'] > 1500) & (data['x'] < 8200)) |
    ((data['x'] > 8200) & (data['y'] > 450) & (data['x'] < 9650)) |
    ((data['x'] > 9650) & (data['y'] > 450) & (data['y'] < 4700) & (data['x'] < 10190))
)
data['in_shop'] = shop_bounds
data['is_out_of_bounds'] = ~data['in_shop'] & ~data['is_charging_station']

in_shop_pts = data['in_shop'].sum()
charging_pts = data['is_charging_station'].sum()
out_pts = data['is_out_of_bounds'].sum()

print(f"Total Points: {len(data)}")
print(f"- Inside Shop (Usable): {in_shop_pts}")
print(f"- Charging Station (0,0): {charging_pts}")
print(f"- Outside Shop (Noise): {out_pts}")

labels = ['Inside Shop', 'Charging Station (0,0)', 'Outside Bounds (Noise)']
sizes = [in_shop_pts, charging_pts, out_pts]
colors = ['#4CAF50', '#FFC107', '#F44336']

plt.figure(figsize=(7, 7))
plt.pie(sizes, labels=labels, autopct='%1.1f%%', colors=colors, startangle=140)
plt.title('Positioning Usability')
plt.show()

## 3. Drift Analysis on Cleaned Data
Let's check the drift vs Q on ONLY the cleaned in-shop data, removing the massive leaps caused by out-of-bounds metrics.

In [ ]:
clean_data = data[data['in_shop']].copy()

clean_data['q_bin'] = pd.qcut(clean_data['q'], q=5, duplicates='drop')
plt.figure(figsize=(10, 6))
sns.boxplot(x='q_bin', y='distance', data=clean_data)
plt.title('Cleaned Data: Boxplot of Distance (Drift) by Quality (q) Bins')
plt.xlabel('q Bins')
plt.ylabel('Distance')
plt.yscale('log')
plt.show()

## 4. Positioning Accuracy using Stationary Devices (Paikannustarkkuus)
We identify periods when the device is stationary (e.g. speed < 5) and calculate the standard deviation of `x` and `y`.

In [ ]:
stationary_data = clean_data[clean_data['speed'] < 5.0].copy()

print(f"Number of Stationary Points: {len(stationary_data)}")
print(f"X Standard Deviation (Accuracy): {stationary_data['x_diff'].std():.2f}")
print(f"Y Standard Deviation (Accuracy): {stationary_data['y_diff'].std():.2f}")
print(f"Average Euclidean Error when stationary: {stationary_data['distance'].mean():.2f}")

# Check standard deviation vs Q
stationary_data['q_rounded'] = stationary_data['q'].round(-1)
std_by_q = stationary_data.groupby('q_rounded').agg({'x_diff': 'std', 'y_diff': 'std'}).reset_index()

plt.figure(figsize=(10, 5))
plt.plot(std_by_q['q_rounded'], std_by_q['x_diff'], label='X Standard Deviation', marker='o')
plt.plot(std_by_q['q_rounded'], std_by_q['y_diff'], label='Y Standard Deviation', marker='s')
plt.title('Positioning Standard Deviation vs Quality (q) [Stationary Devices]')
plt.xlabel('Quality (q)')
plt.ylabel('Standard Deviation (Distance)')
plt.legend()
plt.show()

## 5. Visualizing a Single Visit on the Map
We will extract one coherent visit (session) from the cleaned data and plot it exactly onto the store's physical map layout.

In [ ]:
import matplotlib.image as mpimg

# Load image map
img_path = '../image/kauppa.png'
img = mpimg.imread(img_path)

# Extract robust sessions using 30-minute inactivity threshold
clean_data = clean_data.sort_values(by=['node_id', 'timestamp']).reset_index(drop=True)
clean_data['time_diff'] = clean_data.groupby('node_id')['timestamp'].diff().dt.total_seconds()
clean_data['is_new_session'] = (clean_data['time_diff'] > 1800).astype(int)
clean_data['session_id'] = clean_data.groupby('node_id')['is_new_session'].cumsum()
clean_data['full_session_id'] = clean_data['node_id'].astype(str) + "_" + clean_data['session_id'].astype(str)

# Remove the downstairs charging station and entrance artifacts using radius logic
mask_charge_1 = ((clean_data['x'] - 100)**2 + (clean_data['y'] - 2500)**2) < 400**2
mask_charge_2 = ((clean_data['x'] - 900)**2 + (clean_data['y'] - 3600)**2) < 600**2

# Apply mask and filter extreme signal jumps
visit_data = clean_data[~mask_charge_1 & ~mask_charge_2].copy()
bad_jumps_mask = visit_data['speed'] > 5.0
visit_data = visit_data[~bad_jumps_mask]

# Choose one session with plenty of points
session_counts = visit_data['full_session_id'].value_counts()
valid_session_id = session_counts[session_counts > 100].index[0]
one_visit = visit_data[visit_data['full_session_id'] == valid_session_id]

print(f"Plotted session: {valid_session_id}")
print(f"Data points: {len(one_visit)}")
print(f"Duration: {one_visit['timestamp'].max() - one_visit['timestamp'].min()}")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(24, 10))

# Setup extent [xmin, xmax, ymax, ymin] since image y=0 is at the top
map_extent = [0, 10406, 5220, 0]

# Plot 1: With Scatter Overlay
axs[0].imshow(img, extent=map_extent)
axs[0].plot(one_visit['x'], one_visit['y'], color='blue', linewidth=1.5, alpha=0.5, label='Path Route')
axs[0].scatter(one_visit['x'], one_visit['y'], color='red', s=10, alpha=0.7, label='Datapoints', zorder=3)
axs[0].set_title("One Visit Trace (With Scatter)", fontsize=16)
axs[0].set_xlabel("x (cm) checkout line", fontsize=12)
axs[0].set_ylabel("y (cm)", fontsize=12)
axs[0].legend(loc='lower right')

# Plot 2: Without Scatter Overlay (Lines Only)
axs[1].imshow(img, extent=map_extent)
axs[1].plot(one_visit['x'], one_visit['y'], color='blue', linewidth=2, alpha=0.8, label='Path Route')
axs[1].set_title("One Visit Trace (Without Scatter)", fontsize=16)
axs[1].set_xlabel("x (cm) checkout line", fontsize=12)
axs[1].set_ylabel("y (cm)", fontsize=12)
axs[1].legend(loc='lower right')

plt.tight_layout()
plt.show()

## 6. Visualizing Three Different Visits
Plotting three different robust sessions on the same map to compare their paths.

In [ ]:
# Select three different robust sessions
top_3_sessions = session_counts[session_counts > 100].index[:3]

fig, axs = plt.subplots(1, 2, figsize=(24, 10))
colors = ['blue', 'green', 'orange']

# Display the map background on both axes
axs[0].imshow(img, extent=map_extent)
axs[1].imshow(img, extent=map_extent)

print("Stats for plotted paths:")
for i, session_id in enumerate(top_3_sessions):
    visit = visit_data[visit_data['full_session_id'] == session_id]
    duration = visit['timestamp'].max() - visit['timestamp'].min()
    print(f"{colors[i].capitalize()} path (Session {session_id}): {len(visit)} datapoints, Duration: {duration}")
    
    # Plot 1: With Scatter
    axs[0].plot(visit['x'], visit['y'], color=colors[i], linewidth=1.5, alpha=0.5, label=f'Path {session_id}')
    axs[0].scatter(visit['x'], visit['y'], color=colors[i], s=10, alpha=0.7, zorder=3)
    
    # Plot 2: Without Scatter
    axs[1].plot(visit['x'], visit['y'], color=colors[i], linewidth=2, alpha=0.8, label=f'Path {session_id}')

axs[0].set_title("Three Visit Traces (With Scatter)", fontsize=16)
axs[0].set_xlabel("x (cm) checkout line", fontsize=12)
axs[0].set_ylabel("y (cm)", fontsize=12)
axs[0].legend(loc='lower right')

axs[1].set_title("Three Visit Traces (Without Scatter)", fontsize=16)
axs[1].set_xlabel("x (cm) checkout line", fontsize=12)
axs[1].set_ylabel("y (cm)", fontsize=12)
axs[1].legend(loc='lower right')

plt.tight_layout()
plt.show()

## 7. Visualizing the Raw Data for the Same Three Visits
Here we plot the unfiltered raw data from `data` for the exact same three time periods. This reveals all the noise, GPS jumps, and out-of-bounds artifacts that our filtering removed.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(24, 10))

axs[0].imshow(img, extent=map_extent)
axs[1].imshow(img, extent=map_extent)

print("Plotting pristine RAW data for the 3 visits (plus 5 mins before/after to see arrivals/departures):")
for i, session_id in enumerate(top_3_sessions):
    # Get the time window of the cleaned visit
    visit = visit_data[visit_data['full_session_id'] == session_id]
    target_node = visit['node_id'].iloc[0]
    
    # Expand the window by 5 minutes before and after to see how they arrived/left
    start_time = visit['timestamp'].min() - pd.Timedelta(minutes=5)
    end_time = visit['timestamp'].max() + pd.Timedelta(minutes=5)
    
    # Slice the completely RAW original dataframe 
    raw_visit = data[
        (data['node_id'] == target_node) & 
        (data['timestamp'] >= start_time) & 
        (data['timestamp'] <= end_time)
    ]
    
    print(f"{colors[i].capitalize()} RAW path (Node {target_node}): {len(raw_visit)} raw datapoints vs {len(visit)} clean")
    
    # Plot 1: With Scatter
    axs[0].plot(raw_visit['x'], raw_visit['y'], color=colors[i], linewidth=1.5, alpha=0.5, label=f'Raw Path {session_id}')
    axs[0].scatter(raw_visit['x'], raw_visit['y'], color=colors[i], s=10, alpha=0.7, zorder=3)
    
    # Plot 2: Without Scatter
    axs[1].plot(raw_visit['x'], raw_visit['y'], color=colors[i], linewidth=2, alpha=0.8, label=f'Raw Path {session_id}')

axs[0].set_title("Three Visit Traces (RAW DATA - With Scatter)", fontsize=16)
axs[0].set_xlabel("x (cm) checkout line", fontsize=12)
axs[0].set_ylabel("y (cm)", fontsize=12)
axs[0].legend(loc='lower right')

axs[1].set_title("Three Visit Traces (RAW DATA - Without Scatter)", fontsize=16)
axs[1].set_xlabel("x (cm) checkout line", fontsize=12)
axs[1].set_ylabel("y (cm)", fontsize=12)
axs[1].legend(loc='lower right')

plt.tight_layout()
plt.show()